In [ ]:
!pip install vectorbt

In [ ]:
from vectorbt import *

In [ ]:
from pandas import read_excel

DATA = read_excel( io="Pumpkin Spice Latte.xlsx" , index_col=0 )

CHART = DATA.Close.vbt.range_split( n=45 , range_len=550 , plot=True)
CHART.update_layout( height=800 , width=1000 , template="plotly_dark")

PRICES , DATES = DATA.Close.vbt.range_split( n=45 , range_len=550)

CHART.show()

In [ ]:
def BACKTEST_BB( W , TOP_A , BOTTOM_A , DATA , FEE ) :
  BOTTOM = BBANDS.run( DATA , window=W , alpha=BOTTOM_A )

  TOP = BBANDS.run( DATA , window=W , alpha=TOP_A )

  BUY = BOTTOM.close_crossed_below( BOTTOM.lower )

  SELL = TOP.close_crossed_above( TOP.upper )

  BACKTESTS = Portfolio.from_signals( DATA , entries=BUY , exits=SELL , fees=FEE )

  RETURN = BACKTESTS.total_return().mean()

  BENCH = BACKTESTS.total_benchmark_return().mean()

  return [ W , TOP_A , BOTTOM_A , RETURN , BENCH]

In [ ]:
from joblib.parallel import Parallel
from itertools import product
from numpy import arange

COMBO = product( range(  10 ,  51 ,   1 ) ,
                arange( 1.8 , 5.1 , 0.1 ) ,
                arange( 1.8 , 5.1 , 0.1 ) )

from joblib import delayed , Parallel

TASK = ( delayed( BACKTEST_BB ) ( W , T , B , PRICES.to_numpy() , 0.001 )
                            for ( W , T , B ) in COMBO )

Engine = Parallel(n_jobs=-1)

Results = Engine (TASK)

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning:

A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.



In [ ]:
from pandas import DataFrame
FINDINGS = DataFrame( data = Results ,
columns=["WINDOW" , "UPPER" , "LOWER" , "RETURN" , "BENCHMARK"])

SORTED = FINDINGS.sort_values( "RETURN", ascending=False)
SORTED

,WINDOW,UPPER,LOWER,RETURN,BENCHMARK
34258,41,3.3,2.2,0.055851,0.018581
37559,44,3.4,2.3,0.055226,0.018581
37558,44,3.4,2.2,0.055226,0.018581
37557,44,3.4,2.1,0.054431,0.018581
30956,38,3.2,2.0,0.052090,0.018581
...,...,...,...,...,...
2185,12,1.8,2.5,-0.044841,0.018581
2184,12,1.8,2.4,-0.045568,0.018581
1194,11,2.1,2.4,-0.045710,0.018581
3274,13,1.8,2.5,-0.047942,0.018581
